# 05 - Benchmark final: split test, satu sesi GPU

Membuka split test SATU KALI, memakai konfigurasi terbaik tiap skenario menurut
`best.json`, lalu mengukur inferensi ketiga model berurutan pada GPU yang sama.

Aturan validitas: angka efisiensi (waktu latih, latency, peak memory) hanya sah
bila berasal dari satu hardware dan satu sesi. Menjalankan ulang sebagian
notebook ini di sesi lain lalu mencampur angkanya membatalkan perbandingan.
F1 tidak terpengaruh hardware.

Prasyarat: ketiga skenario sudah punya run di `04_tuning_campaign.ipynb`.

Jangan menjalankan notebook ini berulang kali untuk memilih hasil terbaik: itu
mengubah test menjadi validation set kedua.

In [1]:
import json

import pandas as pd

from src.config import settings
from src.services.campaign import CampaignRunner

OUT_DIR = settings.default_out_dir
runner = CampaignRunner(out_dir=OUT_DIR)

hardware = runner.write_hardware()
print(json.dumps(hardware, indent=2, ensure_ascii=False))

/workspace/indobert-with-rac/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "gpu": "NVIDIA GeForce RTX 3090",
  "cuda_available": true,
  "cuda_version": "13.0",
  "torch": "2.12.1+cu130",
  "transformers": "5.12.1",
  "vram_total_mb": 24124,
  "recorded_at": "2026-09-14 17:38:55",
  "nvidia_smi": "NVIDIA GeForce RTX 3090, 580.178.04, 24576 MiB"
}


## 1. Konfigurasi yang akan diuji

In [2]:
best = json.loads((OUT_DIR / "best.json").read_text(encoding="utf-8"))
for skenario, entri in best.items():
    print(f"{skenario}: run #{entri['run_id']} | val F1 {entri['val_f1_macro']:.4f} "
          f"| {entri['config']}")

rma: run #22 | val F1 0.9832 | {'lr': 3e-05, 'epochs': 8, 'batch': 32, 'warmup_ratio': 0.1, 'weight_decay': 0.01, 'micro_batch': 32, 'seed': 42}
rmb: run #24 | val F1 0.9653 | {'head_arch': 'mlp', 'hidden_dim': 1024, 'epochs': 10, 'lr': 0.001, 'dropout': 0.1, 'weight_decay': 0.0, 'batch': 32, 'seed': 42}
rmc: run #13 | val F1 0.9700 | {'alpha': 0.2, 'k': 1, 'weighting': 'similarity'}


## 2. Jalankan benchmark

In [3]:
hasil = runner.run_final()

2026-09-14 17:38:55,657 | INFO     | src.services.data | Data dimuat: train=6588 val=1402 test=1405 | device=cuda | encoder=indobenchmark/indobert-base-p2
2026-09-14 17:38:56,726 | INFO     | src.services.campaign | Benchmark final dimulai (satu sesi, device cuda)


[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `5`.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 58339.73it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p2
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


2026-09-14 17:38:58,036 | INFO     | src.models.heads | Inisialisasi embedding special token: 3/3


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 63088.93it/s]

2026-09-14 17:38:59,007 | INFO     | src.models.heads | Inisialisasi embedding special token: 3/3
2026-09-14 17:38:59,074 | INFO     | src.services.features | Fitur beku dimuat dari cache /workspace/indobert-with-rac/outputs/tuning/features/indobenchmark__indobert-base-p2


2026-09-14 17:39:00,577 | INFO     | src.services.rac | Indeks FAISS dibangun: 6588 vektor berdimensi 768
2026-09-14 17:39:00,933 | INFO     | src.services.rac | Indeks FAISS dibangun: 6588 vektor berdimensi 768
2026-09-14 17:39:01,479 | INFO     | src.services.campaign | [final] RM-a: 4.798 ms/sampel | peak 1049 MB
2026-09-14 17:39:02,006 | INFO     | src.services.campaign | [final] RM-b: 4.586 ms/sampel | peak 856 MB
2026-09-14 17:39:02,638 | INFO     | src.services.campaign | [final] RM-c: 5.734 ms/sampel | peak 856 MB
2026-09-14 17:39:02,653 | INFO     | src.services.campaign | [final] perbandingan:
model                                                                                                                            config  test_f1_macro  test_acc  test_f1_judi  test_precision_judi  test_recall_judi  val_f1_macro  trainable_params  train_time_s  infer_latency_ms  infer_peak_gpu_mem_mb
 RM-a                 {"lr": 3e-05, "epochs": 8, "batch": 32, "warmup_ratio": 0.1, "weig

## 3. Perbandingan test

In [4]:
hasil["comparison"]

,model,config,test_f1_macro,test_acc,test_f1_judi,test_precision_judi,test_recall_judi,val_f1_macro,trainable_params,train_time_s,infer_latency_ms,infer_peak_gpu_mem_mb
0,RM-a,"{""lr"": 3e-05, ""epochs"": 8, ""batch"": 32, ""warmu...",0.967709,0.980783,0.947162,0.949020,0.945312,0.983223,109485314,132.18,4.7985,1049.5
1,RM-b,"{""head_arch"": ""mlp"", ""hidden_dim"": 1024, ""epoc...",0.948574,0.969395,0.915851,0.917647,0.914062,0.965301,789506,9.03,4.5857,856.4
2,RM-c,"{""alpha"": 0.2, ""k"": 1, ""weighting"": ""similarity""}",0.950816,0.970819,0.919450,0.924901,0.914062,0.969995,0,0.00,5.7336,856.4


## 4. Benchmark inferensi

In [5]:
hasil["benchmark"]

,scenario,infer_latency_ms,infer_peak_gpu_mem_mb
0,RM-a,4.7985,1049.5
1,RM-b,4.5857,856.4
2,RM-c,5.7336,856.4


Ketiga skenario menjalankan forward pass encoder 110 juta parameter yang sama
saat inferensi, sehingga latency-nya diperkirakan berdekatan. Efisiensi RM-b dan
RM-c terletak pada jumlah parameter yang dilatih dan waktu latih, bukan pada
kecepatan prediksi. Kalau angkanya memang begitu, itu temuan yang harus
dinyatakan apa adanya di Bab 4, bukan disembunyikan.

## 5. Kriteria sukses

In [6]:
hasil["criteria"]

,model,f1_gap_pp,lolos_f1_gap<=3pp,param_reduction_pct,lolos_param>=90%,train_time_s,rma_train_time_s,time_reduction_pct,lolos_waktu>=50%,kriteria_terpenuhi,kompetitif(>=2/3)
0,RM-b,1.91,True,99.2789,True,9.03,132.18,93.17,True,3/3,True
1,RM-c,1.69,True,100.0000,True,0.00,132.18,100.00,True,3/3,True


Strategi ringan dianggap kompetitif bila memenuhi minimal dua dari tiga syarat:
selisih F1 tidak lebih dari 3 poin persentase, pengurangan trainable parameter
minimal 90%, dan pengurangan waktu latih minimal 50%.

## 6. Ekspor checkpoint final

In [7]:
import shutil

settings.model_dir.mkdir(parents=True, exist_ok=True)
for nama in ("rma_best.pt", "rmb_best.pt", "rmc_best.pt"):
    sumber = OUT_DIR / "checkpoints" / nama
    if sumber.exists():
        tujuan = shutil.copy2(sumber, settings.model_dir / nama)
        print(f"  {nama}: {sumber.stat().st_size / 1024**2:.1f} MB -> {tujuan}")

  rma_best.pt: 417.7 MB -> /workspace/indobert-with-rac/models/rma_best.pt
  rmb_best.pt: 3.0 MB -> /workspace/indobert-with-rac/models/rmb_best.pt
  rmc_best.pt: 0.0 MB -> /workspace/indobert-with-rac/models/rmc_best.pt


## Ringkasan

Tabel di atas adalah angka resmi Bab 4. Seluruhnya diproduksi dalam satu sesi
pada hardware yang tercatat di `hardware.json` yang berdampingan dengannya.

Lanjut ke `06_analysis_export.ipynb`.